### Load packages

In [445]:
import pandas as pd
import numpy as np
import os
# import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

### Define parameters and file info

In [446]:
pd.set_option('display.max_colwidth', None)

In [447]:
data_directory = r"..\data"

# -----------------------------------------------------------------------
# CSV data
# -----------------------------------------------------------------------

node_file = r"MCN_nonprofit_economy_revenue_nodes_2023.csv"
node_path = os.path.join(data_directory, node_file)
print("Node CSV file:", node_path)
if os.path.exists(node_path):
    print("EXISTS")

edge_file = r"MCN_nonprofit_economy_revenue_edges_2023.csv"
edge_path = os.path.join(data_directory, edge_file)
print("Edge CSV file:", edge_path)
if os.path.exists(edge_path):
    print("EXISTS")

# -----------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------

fig_output_file = "MCN_nonprofit_economy_sankey_2023"
output_directory = r"..\output"

Node CSV file: ..\data\MCN_nonprofit_economy_revenue_nodes_2023.csv
EXISTS
Edge CSV file: ..\data\MCN_nonprofit_economy_revenue_edges_2023.csv
EXISTS


### Load data

In [448]:
# Load nonprofit economy node data

node_raw_df = pd.read_csv(node_path)
node_raw_df.head()

,Node,Node Type,X,Y,X Offset,Y Offset
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.600,0.0400,0.032,-0.007
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.630,0.0375,0.000,0.000
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.330,0.0800,0.025,-0.010
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.355,0.0735,0.000,0.000
4,Program fees from private sources,Source,0.100,0.8600,0.020,-0.037


In [449]:
# Load nonprofit economy edge data

edge_raw_df = pd.read_csv(edge_path)
edge_raw_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


In [450]:
# Clean & format columns in edge data

edge_clean_df = edge_raw_df.copy()

edge_clean_df["Amount"] = pd.to_numeric(
    edge_clean_df["Amount"].replace("-", 0),
    errors="coerce"
)

edge_clean_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


### Clean up data in dataframes

In [451]:
node_raw_df.head()

,Node,Node Type,X,Y,X Offset,Y Offset
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.600,0.0400,0.032,-0.007
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.630,0.0375,0.000,0.000
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.330,0.0800,0.025,-0.010
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.355,0.0735,0.000,0.000
4,Program fees from private sources,Source,0.100,0.8600,0.020,-0.037


In [452]:
# Function for setting node totals

def get_total(row):
    node = row["Node"]
    node_type = row["Node Type"]

    if node_type == "Source":
        amount = edge_clean_df.loc[
            edge_clean_df["Source"] == node,
            "Amount"
        ].sum()

        return f"${amount:,.0f}B"

    elif node_type == "Recipient":
        amount = edge_clean_df.loc[
            edge_clean_df["Recipient"] == node,
            "Amount"
        ].sum()

        return f"${amount:,.0f}B"

    elif node_type == "Intermediary Right":
        incoming = edge_clean_df.loc[
            edge_clean_df["Recipient"] == node,
            "Amount"
        ].sum()

        outgoing = edge_clean_df.loc[
            edge_clean_df["Source"] == node,
            "Amount"
        ].sum()

        return f"In: ${incoming:,.0f}B<br>Out: ${outgoing:,.0f}B"

    return ""

In [453]:
# Add column for total in or out of node

node_totals_df = node_raw_df.copy()

node_totals_df["Total"] = node_totals_df.apply(get_total, axis=1)

node_totals_df.head(10)

,Node,Node Type,X,Y,X Offset,Y Offset,Total
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.600,0.0400,0.032,-0.0070,In: $76B<br>Out: $54B
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.630,0.0375,0.000,0.0000,
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.330,0.0800,0.025,-0.0100,In: $178B<br>Out: $104B
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.355,0.0735,0.000,0.0000,
4,Program fees from private sources,Source,0.100,0.8600,0.020,-0.0370,"$1,923B"
5,Federal government,Source,0.100,0.7700,0.020,-0.0250,$469B
6,State and local government,Source,0.100,0.6800,0.020,-0.0280,$204B
7,Federated Giving,Source,0.100,0.5900,0.020,-0.0012,$3B
8,Investment Income,Source,0.100,0.5000,0.020,-0.0005,$67B
9,Corporations,Source,0.100,0.4100,0.020,0.0010,$37B


In [454]:
# Function to find the middle of a string and add a <br> at the space that is closest to that wrap_at_middle

def wrap_at_middle(text):
    if pd.isna(text):
        return text
    
    # Find positions of all spaces
    space_positions = [i for i, char in enumerate(text) if char == " "]
    
    # If no spaces, return unchanged
    if not space_positions:
        return text
    
    # Midpoint of string
    midpoint = len(text) / 2
    
    # Find space closest to midpoint
    split_pos = min(space_positions, key=lambda x: abs(x - midpoint))
    
    # Insert <br>
    if "DAF" in text:
        wrapped_text = text
    else:
        wrapped_text = text[:split_pos] + "<br>" + text[split_pos + 1:]

    return wrapped_text

In [455]:
# add extra columns for node name formatting
node_colname_df = node_totals_df.copy()

node_colname_df["Node Short"] = node_colname_df["Node"].replace({
    "Donor-advised fund sponsors (national and community foundation only)": "DAF sponsors",
    "Donor-advised fund sponsors (national and community foundation only) (ghost)": "DAF sponsors (ghost)",
    "Foundations (minus community foundation DAF sponsors)": "Foundations",
    "Foundations (minus community foundation DAF sponsors) (ghost)": "Foundations (ghost)",
    "Program fees from private sources": "Program fees",
    "State and local government": "State & local government",
    "Federated Giving": "Federated giving",
    "Hospitals and nursing homes": "Hospitals & nursing homes",
    "Health (minus hospitals and nursing homes)": "Other healthcare",
    "Education (minus colleges and universities)": "Other education",
    "Public/societal benefit (minus national DAF sponsors)": "Public/societal benefit",
    "Arts, culture, humanities": "Arts & culture",
    "Environment and animals": "Environment & animals",
    "International/foreign affairs": "International/ foreign affairs",
    "Unknown, unclassified": "Unknown"
})

In [456]:
# Create a different node text columns for different needs

node_format_df = node_colname_df.copy()

# Create Node Wrapped column
node_format_df["Node Wrapped"] = node_format_df["Node Short"].apply(wrap_at_middle)

# Create Node Hover column
node_format_df["Node Hover"] = node_format_df["Node Short"].str.replace(" (ghost)", "", regex=False)

# Create Node Totals column

conditions = [
    node_format_df["Node Type"] == "Source",
    node_format_df["Node Type"] == "Recipient",
    node_format_df["Node Type"].isin(["Intermediary Right", "Intermediary Left"])
]

choices = [
    # "From<br>" + node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str),
    "From<br>" + (node_format_df["Node Wrapped"].str[:1].str.lower() + node_format_df["Node Wrapped"].str[1:]) + "<br>" + node_format_df["Total"].astype(str),
    # "To<br>" + node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str),
    "To<br>" + (node_format_df["Node Wrapped"].str[:1].str.lower() + node_format_df["Node Wrapped"].str[1:]) + "<br>" + node_format_df["Total"].astype(str),
    
    node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str)
]

node_format_df["Node Totals"] = np.select(
    conditions,
    choices,
    default=node_format_df["Node Wrapped"]
)

node_format_df.head(20)

,Node,Node Type,X,Y,X Offset,Y Offset,Total,Node Short,Node Wrapped,Node Hover,Node Totals
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.600,0.0400,0.032,-0.0070,In: $76B<br>Out: $54B,DAF sponsors,DAF sponsors,DAF sponsors,DAF sponsors<br>In: $76B<br>Out: $54B
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.630,0.0375,0.000,0.0000,,DAF sponsors (ghost),DAF sponsors (ghost),DAF sponsors,DAF sponsors (ghost)
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.330,0.0800,0.025,-0.0100,In: $178B<br>Out: $104B,Foundations,Foundations,Foundations,Foundations<br>In: $178B<br>Out: $104B
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.355,0.0735,0.000,0.0000,,Foundations (ghost),Foundations<br>(ghost),Foundations,Foundations<br>(ghost)
4,Program fees from private sources,Source,0.100,0.8600,0.020,-0.0370,"$1,923B",Program fees,Program<br>fees,Program fees,"From<br>program<br>fees<br>$1,923B"
5,Federal government,Source,0.100,0.7700,0.020,-0.0250,$469B,Federal government,Federal<br>government,Federal government,From<br>federal<br>government<br>$469B
6,State and local government,Source,0.100,0.6800,0.020,-0.0280,$204B,State & local government,State & local<br>government,State & local government,From<br>state & local<br>government<br>$204B
7,Federated Giving,Source,0.100,0.5900,0.020,-0.0012,$3B,Federated giving,Federated<br>giving,Federated giving,From<br>federated<br>giving<br>$3B
8,Investment Income,Source,0.100,0.5000,0.020,-0.0005,$67B,Investment Income,Investment<br>Income,Investment Income,From<br>investment<br>Income<br>$67B
9,Corporations,Source,0.100,0.4100,0.020,0.0010,$37B,Corporations,Corporations,Corporations,From<br>corporations<br>$37B


### Prepare data for plotting

In [457]:
# Select node field to use

node_display_field = "Node Totals"
node_id_field = "Node"

In [458]:
# Prepare data for plotting

# Build a node map (convert node labels into integer indices)
node_map = {
    node: i for i, node in enumerate(node_format_df[node_id_field])
}

# Convert source and recipient node into indices
sources = edge_clean_df["Source"].map(node_map)
targets = edge_clean_df["Recipient"].map(node_map)
values = edge_clean_df["Amount"]

### Colors

In [459]:
# Define colors and transparency

transparency = 0.35

daf_color = f"rgba(140,0,0,{transparency})"
pf_color = f"rgba(0,0,160,{transparency})"

source_colors = [
    (160, 160, 160),  # gray
    (230, 110, 110),  # pink
    (230, 170, 110),  # orange
    (230, 210, 110),  # yellow
    (160, 205, 120),  # green
    (120, 190, 170),  # teal
    (110, 170, 230),  # blue
    (170, 110, 230),  # purple
]

category_palette_index = {
    "Program fees from private sources": 0,
    "Federal government": 1,
    "State and local government": 2,
    "Federated Giving": 3,
    "Investment Income": 4,
    "Corporations": 5,
    "Bequests": 6,
    "Individuals": 7,
}

In [460]:
# Function to turn colors into RGBA values
def to_rgba(rgb, alpha=transparency):
    return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{alpha})"

In [461]:
# Function to map a color to the right edge or node by index
def get_color(node):
    if "Donor-advised fund" in node:
        return daf_color

    if "Foundations" in node:
        return pf_color

    idx = category_palette_index.get(node, 0)
    return to_rgba(source_colors[idx])

In [462]:
# Map colors to edges and nodes

edge_colors = edge_clean_df["Source"].map(get_color)

node_colors = node_format_df["Node"].apply(get_color)

# Set all recipient nodes to medium gray
node_colors[node_format_df["Node Type"] == "Recipient"] = "rgb(160,160,160)"

### Prepare node and hover text

In [463]:
# Set font family

font_family = "Source Sans 3, Helvetica Neue, Arial, sans-serif"
# font_family = "Inter, Segoe UI, Roboto, Arial, sans-serif"
# font_family = "Arial, Helvetica, sans-serif"

# Build a CSS-safe version of the font_family variable

font_family_css = ", ".join(
    f'"{f.strip()}"' if " " in f.strip() else f.strip()
    for f in font_family.split(",")
)

In [464]:
# Create custom node names 
# node_names = node_format_df[node_display_field].tolist()
hover_node_names = node_format_df["Node Hover"].tolist()

# Build hover display fields
hover_text = []

for s, t, v in zip(sources, targets, values):
    source_name = hover_node_names[s]
    target_name = hover_node_names[t]

    if "DAF" in target_name:
        hover_target_name = target_name
    else:
        hover_target_name = target_name[0].lower() + target_name[1:] # lowercase first letter in target name

    hover_text.append(
        f'<div style="font-size: 12px; margin: 0;">'
        f'{source_name} to</br>{target_name}'
        f'</div>'
        f'<div style="font-size: 16px; margin-top: 4px;">'
        f'<b>${v:,.1f}B</b>'
        f'</div>'
    )

### Visualize with Plotly Sankey (static charts)

Using Plotly Sankey because it includes the best compromises for:
- weighted edges
- self-loops (still poor)
- curved edges
- label control (still poor)
- heirarchy levels
- interactivity
- quality
- merging flows

In [465]:
# Plot the diagram
fig = go.Figure(go.Sankey(
    arrangement="snap",
    
    node=dict(
        # label=node_format_df["Node"],
        label=[""] * len(node_format_df), # remove default labels
        x=node_format_df["X"],
        y=node_format_df["Y"],
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        # line=dict(color="rgba(0,0,0,0)", width=0),
        color=node_colors,
        hoverinfo="skip"
    ),
    
    link=dict(
        # arrowlen=10,
        source=sources,
        target=targets,
        value=values,
        color=edge_colors,
        customdata=hover_text,
        hoverinfo="none"
        # hovertemplate="%{customdata}<extra></extra>"
    )
))

# Define annotations for labels
annotations = []

for _, row in node_format_df[node_format_df["Node Type"] == "Source"].iterrows():
    node_name = row[node_id_field]
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] - dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="right",
        align="center",
        textangle=-90
    ))

for _, row in node_format_df[node_format_df["Node Type"] == "Recipient"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] + dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="left",
        align="center",
        textangle=-90
    ))
for _, row in node_format_df[node_format_df["Node Type"] == "Intermediary Left"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] - dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="left",
        align="left",
        textangle=-90
    ))

for _, row in node_format_df[node_format_df["Node Type"] == "Intermediary Right"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] + dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="right",
        align="right",
        textangle=-90
    ))

# Format
fig.update_layout(
    # title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20),
    font=dict(family=font_family, size=11, color="black"),
    annotations=annotations
)

fig.show()

### Output image

In [466]:

# Output as PNG
fig_png_output_path = os.path.join(output_directory, fig_output_file + ".png")
# fig.write_image(fig_output_path)

# Output as SVG
fig_svg_output_path = os.path.join(output_directory, fig_output_file + ".svg")
# fig.write_image(fig_svg_output_path)

In [467]:
# Output to HTML

fig_html_output_path = os.path.join(output_directory, fig_output_file + ".html")

# export as internet-required html page
# html_fragment = pio.to_html(fig, full_html=False, include_plotlyjs="cdn")

# export as fully standalone, no-internet-required html page
html_fragment = pio.to_html(fig, full_html=False, include_plotlyjs=True)

full_html = f"""
<html>
<head>
<meta charset="utf-8" />
<style>
.rotated-wrapper {{
    position: relative;
    width: 1200px;
    height: 800px;
}}

.rotated-plot {{
    position: absolute;
    top: 0;
    left: 0;
    transform: rotate(90deg) translateY(-100%) translateX(0px);
    transform-origin: top left;
}}

#custom-tooltip {{
    font-family: {font_family_css};
    font-size: 8px;
    position: absolute;
    display: none;
    background: white;
    border: 1px solid #999;
    padding: 4px;
    font-size: 12px;
    pointer-events: none;
    z-index: 9999;
    box-shadow: 2px 2px 6px rgba(0,0,0,0.2);
}}

</style>
</head>
<body>

<div class="rotated-wrapper">
<div class="rotated-plot">
{html_fragment}
</div>
</div>

<div id="custom-tooltip"></div>

<script>
window.addEventListener("load", function() {{

    const tooltip = document.getElementById("custom-tooltip");
    const plotDiv = document.querySelector(".plotly-graph-div");

    if (!plotDiv) return;

    plotDiv.on("plotly_hover", function(eventData) {{

        const point = eventData.points[0];
        const e = eventData.event;

        tooltip.innerHTML = point.customdata;
        tooltip.style.display = "block";

        tooltip.style.left = (e.pageX + 15) + "px";
        tooltip.style.top = (e.pageY + 15) + "px";
    }});

    plotDiv.on("plotly_unhover", function() {{

        tooltip.style.display = "none";

    }});

}});
</script>

</body>
</html>
"""

with open(fig_html_output_path, "w", encoding="utf-8") as f:
    f.write(full_html)